# 13 - VideoMAE fine-tune on all train clips, test snapshot after every epoch (Kaggle GPU)

Same recipe as `12_videomae_finetune` (augmentations against shortcuts, BCE + twin ranking loss, Kinetics + SSv2 backbones, one per T4), but:
- trains on all groups except a 30-group bug-check holdout, 3 epochs;
- saves 4-view TTA test predictions after **every** epoch.

**Why:** locally (1 epoch) VideoMAE alone scored proxy 0.70-0.71 (too weak alone) but its errors are opposite to the VLMs'
(it keys on sudden motion peaks, the VLMs on overall busy motion), so it is a blend partner. Longer train fitting can transfer worse,
so the epoch is picked later with the label-free proxy (`scripts/proxy_eval.py`), never with the LB.
**Output:** `vmae_<tag>_ep<k>_test.csv`, `vmae_ep<k>_test.csv` (rank average of both backbones per epoch).

In [ ]:
import glob, os, sys, json, subprocess, time
import numpy as np, pandas as pd, torch
from scipy.stats import rankdata

found = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("../../data/train_labels.csv")
assert found, "train_labels.csv not found - attach the competition / check the path"
ROOT = os.path.dirname(found[0])
SMOKE = int(os.environ.get("SMOKE", 0))
on_kaggle = os.path.exists("/kaggle/working")
OUT = "/kaggle/working/features" if on_kaggle else "../../artifacts/vmae_smoke"
os.makedirs(OUT, exist_ok=True)
BACKBONES = [("MCG-NJU/videomae-base-finetuned-kinetics", 42), ("MCG-NJU/videomae-base-finetuned-ssv2", 43)]
n_gpu = max(1, torch.cuda.device_count())
NW = max(1, (os.cpu_count() or 2) // max(1, n_gpu))     # dataloader workers per training process
print(ROOT, "| GPUs:", n_gpu, "| loader workers each:", NW, "| torch", torch.__version__)

## The training script
One copy per GPU (one backbone each): 3 epochs, holdout AUC / twin accuracy per epoch, 4-view test snapshot per epoch.

In [ ]:
WORKER = r"""
import sys, os, io, json, time, math
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from transformers import VideoMAEForVideoClassification

NF, S = 16, 224                       # 16 frames, 224x224 (VideoMAE-base input)
TOP, BOT = 0.08, 0.10                 # top/bottom bands blanked in train AND test: timestamps, watermarks, badges
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def load_frames(root, split, cid, idx, jpeg_q=None):
    out = []
    for j in idx:
        im = Image.open(f"{root}/{split}/{cid}/frame_{int(j):03d}.jpg").convert("RGB")
        if jpeg_q:                    # re-compress: JPEG artefacts must not identify the source
            b = io.BytesIO(); im.save(b, "JPEG", quality=jpeg_q); b.seek(0); im = Image.open(b).convert("RGB")
        out.append(np.asarray(im))
    x = torch.from_numpy(np.stack(out)).permute(0, 3, 1, 2).float() / 255      # (T,3,H,W) in [0,1]
    H = x.shape[2]
    x[:, :, :int(TOP * H)] = MEAN; x[:, :, H - int(BOT * H):] = MEAN               # mean colour = 0 after normalising
    return x


def gauss_blur(x, sigma):
    k = int(2 * round(2 * sigma) + 1); r = torch.arange(k).float() - k // 2
    g = torch.exp(-r ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    x = F.conv2d(F.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k).repeat(3, 1, 1, 1), groups=3)
    return F.conv2d(F.pad(x, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1).repeat(3, 1, 1, 1), groups=3)


def finish(x):                        # (T,3,h,w) [0,1] -> (T,3,S,S) normalised
    x = F.interpolate(x, size=(S, S), mode="bilinear", align_corners=False, antialias=True)
    return (x.clamp(0, 1) - MEAN) / STD


def train_view(root, cid, rng):
    # temporal: random crop of the clip, identical policy for both classes (no label leak)
    s, e = (0, 29) if rng.random() < 0.5 else (int(rng.integers(0, 7)), int(rng.integers(23, 30)))
    idx = np.round(np.linspace(s, e, NF)).astype(int)
    if rng.random() < 0.25:           # content moves earlier, last frame held: breaks "crash is always late"
        k = int(rng.integers(1, 5)); idx = np.concatenate([idx[k:], np.repeat(idx[-1], k)])
    if rng.random() < 0.3:            # stutter, like the duplicated frames of the test frame-rate conversion
        for i in range(1, NF):
            if rng.random() < 0.2: idx[i] = idx[i - 1]
    q = int(rng.integers(30, 90)) if rng.random() < 0.25 else None
    x = load_frames(root, "train", cid, idx, q)
    _, _, H, W = x.shape
    h, w = int(H * rng.uniform(0.7, 1)), int(W * rng.uniform(0.7, 1))
    y0, x0 = int(rng.integers(0, H - h + 1)), int(rng.integers(0, W - w + 1))
    x = x[:, :, y0:y0 + h, x0:x0 + w]
    if rng.random() < 0.5: x = x.flip(-1)                                   # horizontal flip ok; never reverse time
    x = x * rng.uniform(0.7, 1.3)
    m = x.mean(dim=(1, 2, 3), keepdim=True); x = (x - m) * rng.uniform(0.7, 1.3) + m
    g = x.mean(1, keepdim=True); x = (x - g) * rng.uniform(0.7, 1.3) + g
    r = rng.random()
    if r < 0.25: x = gauss_blur(x, rng.uniform(0.5, 1.5))
    elif r < 0.6: x = x + rng.uniform(0.5, 1.5) * (x - gauss_blur(x, 1.0))  # sharpen: test is sharper than train
    x = finish(x)
    if rng.random() < 0.25:           # random erasing, same box in every frame
        eh, ew = int(S * rng.uniform(0.1, 0.3)), int(S * rng.uniform(0.1, 0.3))
        ey, ex = int(rng.integers(0, S - eh)), int(rng.integers(0, S - ew)); x[:, :, ey:ey + eh, ex:ex + ew] = 0
    return x


TEST_VIEWS = [(0, 29, False), (0, 29, True), (1, 28, False), (1, 28, True)]   # (start, end, hflip)


def eval_views(root, split, cid, views):
    out = []
    for s, e, fl in views:
        x = load_frames(root, split, cid, np.round(np.linspace(s, e, NF)).astype(int))
        out.append(finish(x.flip(-1) if fl else x))
    return torch.stack(out)


class PairDS(Dataset):
    def __init__(self, root, pairs): self.root, self.pairs = root, pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        rng = np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,))))
        pos, neg = self.pairs[i]
        return torch.stack([train_view(self.root, pos, rng), train_view(self.root, neg, rng)]), torch.tensor([1.0, 0.0])


class EvalDS(Dataset):
    def __init__(self, root, items, views): self.root, self.items, self.views = root, items, views
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return eval_views(self.root, *self.items[i], self.views)


@torch.no_grad()
def predict(model, dl, dev):
    model.eval(); out = []
    for xb in dl:
        B, V = xb.shape[:2]
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(pixel_values=xb.view(B * V, NF, 3, S, S).to(dev)).logits.float().view(B, V)
        out.append(lg.mean(1).cpu())
    return torch.cat(out).numpy()


def main():
    root, backbone, seed, out_dir, smoke, nw = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4], int(sys.argv[5]), int(sys.argv[6])
    tag = backbone.split("-")[-1]
    torch.manual_seed(seed); np.random.seed(seed)
    dev = "cuda"
    EPOCHS, BS = (1, 1) if smoke else (3, 4)                                  # BS = pairs per step (8 clips)
    lab = pd.read_csv(f"{root}/train_labels.csv")
    test_ids = pd.read_csv(f"{root}/sample_submission.csv")["clip_id"].tolist()
    groups = np.random.default_rng(42).permutation(sorted(lab.group_id.unique()))
    n_val = 4 if smoke else 30                                                 # tiny holdout (60 clips) = bug check only; train CV does not predict LB
    val_g, tr_g = set(groups[:n_val]), groups[n_val:]
    if smoke: tr_g = tr_g[:6]; test_ids = test_ids[:6]
    pc = lab.pivot_table(index="group_id", columns="label", values="clip_id", aggfunc="first")
    pairs = [(pc.loc[g, 1], pc.loc[g, 0]) for g in tr_g]
    val = lab[lab.group_id.isin(val_g)].reset_index(drop=True)

    model = VideoMAEForVideoClassification.from_pretrained(backbone, num_labels=1, ignore_mismatched_sizes=True)
    model.gradient_checkpointing_enable(); model.to(dev)
    head = [p for n, p in model.named_parameters() if n.startswith("classifier") or n.startswith("fc_norm")]
    body = [p for n, p in model.named_parameters() if not (n.startswith("classifier") or n.startswith("fc_norm"))]
    opt = torch.optim.AdamW([{"params": body, "lr": 3e-5}, {"params": head, "lr": 1e-3}], weight_decay=0.05)
    dl = DataLoader(PairDS(root, pairs), batch_size=BS, shuffle=True, num_workers=nw, drop_last=True, pin_memory=True, persistent_workers=nw > 0)
    total = EPOCHS * len(dl); warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: min(1, (t + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, t / total))))
    scaler = torch.amp.GradScaler()
    vdl = DataLoader(EvalDS(root, [("train", c) for c in val.clip_id], TEST_VIEWS[:2]), batch_size=4, num_workers=nw)
    tdl = DataLoader(EvalDS(root, [("test", c) for c in test_ids], TEST_VIEWS), batch_size=4, num_workers=nw)
    print(tag, "| train pairs", len(pairs), "| val clips", len(val), "| steps", total, flush=True)

    t0, step = time.time(), 0
    for ep in range(EPOCHS):
        model.train(); run = []
        for xb, yb in dl:
            xb = xb.view(-1, NF, 3, S, S).to(dev, non_blocking=True); yb = yb.view(-1).to(dev)
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(pixel_values=xb).logits.squeeze(-1)
            lg = lg.float()
            # BCE + pairwise ranking inside each twin pair (same scene: only the crash event differs)
            loss = F.binary_cross_entropy_with_logits(lg, yb) + F.softplus(-(lg[0::2] - lg[1::2])).mean()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            run.append(loss.item())
            if step % 50 == 0 or step <= 2:
                print(f"{tag} ep{ep} step {step}/{total} loss {np.mean(run[-50:]):.4f} | {time.time() - t0:.0f}s", flush=True)
        pv = predict(model, vdl, dev)
        v = val.assign(s=pv); tw = v.pivot_table(index="group_id", columns="label", values="s").dropna()
        auc = roc_auc_score(v.label, v.s) if v.label.nunique() > 1 else float("nan")
        print(f"{tag} epoch {ep} | val AUC {auc:.4f} | twin acc {(tw[1] > tw[0]).mean():.3f} | {time.time() - t0:.0f}s", flush=True)
        # test snapshot after EVERY epoch: the epoch is chosen later with the label-free proxy (more train fitting can transfer worse)
        pd.DataFrame({"clip_id": test_ids, "score": predict(model, tdl, dev)}).to_csv(f"{out_dir}/vmae_{tag}_ep{ep}_test.csv", index=False)
        print(f"{tag} epoch {ep} test snapshot saved | {time.time() - t0:.0f}s", flush=True)
    print(tag, "done", f"{time.time() - t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()
"""
open("worker.py", "w").write(WORKER)

## Run both backbones in parallel (one per GPU)

In [ ]:
procs = []
for g, (bb, seed) in enumerate(BACKBONES[:n_gpu] if n_gpu > 1 else BACKBONES):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g % n_gpu))
    log = open(f"worker_{g}.log", "w")
    p = subprocess.Popen([sys.executable, "worker.py", ROOT, bb, str(seed), OUT, str(SMOKE), str(NW)], env=env, stdout=log, stderr=subprocess.STDOUT)
    if n_gpu == 1: p.wait()                                  # one GPU: run them one after the other
    procs.append(p)
t0 = time.time()
for p in procs: p.wait()
print("finished in", f"{time.time() - t0:.0f}s", "| return codes:", [p.returncode for p in procs])
for g in range(len(procs)):
    print(f"--- worker {g} ---"); print("".join(l for l in open(f"worker_{g}.log").readlines() if "epoch" in l or "done" in l or "Error" in l or "error" in l)[-3000:])

## Merge: rank average of the two backbones

In [ ]:
for ep in range(3):
    fs = sorted(glob.glob(f"{OUT}/vmae_*_ep{ep}_test.csv"))
    if not fs: continue
    dfs = [pd.read_csv(f) for f in fs]
    if len(dfs) == 2: print(f"epoch {ep}: test rank corr between backbones", round(np.corrcoef(rankdata(dfs[0].score), rankdata(dfs[1].score))[0, 1], 3))
    avg = np.mean([rankdata(d.score) / len(d) for d in dfs], axis=0)
    pd.DataFrame({"clip_id": dfs[0].clip_id, "score": avg}).to_csv(f"{OUT}/vmae_ep{ep}_test.csv", index=False)
print(sorted(os.listdir(OUT)))